In [1]:
from pathlib import Path

from qdk_chemistry.data import Structure

# Read molecular structure from XYZ file
structure = Structure.from_xyz_file(
    Path(".") / "data/benzene_diradical.structure.xyz"
)

## Generating the molecular orbitals

This step performs a [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method) (HF) SCF calculation to generate an approximate initial wavefunction and ground-state energy guess.
The wavefunction and energy returned by this initial calculation do not provide an accurate description of the system electronic structure; however, they are useful for constructing molecular orbitals.
The resulting molecular orbitals will be used in subsequent steps for active space selection and multi-configuration calculations.

In [2]:
from qdk_chemistry.algorithms import create

# Perform an SCF calculation, returning the energy and wavefunction
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
print(f"SCF energy is {E_hf:.3f} Hartree")

# Display a summary of the molecular orbitals obtained from the SCF calculation
print("SCF Orbitals:\n", wfn_hf.get_orbitals().get_summary())

[2026-04-02 17:15:23.081763] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] mol: atoms=10, electrons=40, n_ecp_electrons=0, charge=0, multiplicity=1, spin(2S)=0, alpha=20, beta=20
[2026-04-02 17:15:23.081813] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] restricted=true, basis=cc-pvdz, pure=true, num_atomic_orbitals=104, density_threshold=1.00e-05, og_threshold=1.00e-07
[2026-04-02 17:15:23.081816] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] fock_alg=TradJK
[2026-04-02 17:15:23.081818] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] eri_threshold=1.00e-12, shell_pair_threshold=1.00e-12
[2026-04-02 17:15:23.081821] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] world_size=1, omp_get_max_threads=8
[2026-04-02 17:15:23.238719] [info] [qdk:chemistry:algorithms:microsoft:scf:src:scf:scf_impl] Reset incremental Fock matrix
[2026-04-02 17:15:23.487542] [info] [qdk:chemistry:algorithms:microsoft:scf:sr

## Selecting an active space and calculating the multi-configuration wavefunction

### Active space selection

Most chemistry applications on quantum computers will require the use of [active spaces](https://en.wikipedia.org/wiki/Complete_active_space) to focus the quantum calculation on a subset of the electrons and orbitals in the system.
For example, the benzene diradical with the default basis set specified above results in ~100 molecular orbitals, requiring ~200 qubits to represent the full electronic structure problem.

This cell shows how to optimize this calculation by selecting an active space from the valence molecular orbitals calculated in the previous SCF step, focusing on the [frontier orbitals](https://en.wikipedia.org/wiki/Frontier_molecular_orbital_theory) that are most relevant to molecular reactivity.

In [3]:
# Select active space (6 electrons in 6 orbitals for benzene diradical) to choose most chemically relevant orbitals
active_space_selector = create("active_space_selector", algorithm_name="qdk_valence",
                               num_active_electrons=6, num_active_orbitals=6)
active_wfn = active_space_selector.run(wfn_hf)
active_orbitals = active_wfn.get_orbitals()

# Print a summary of the active space orbitals
print("Active Space Orbitals:\n", active_orbitals.get_summary())

Active Space Orbitals:
 Orbitals Summary:
  AOs: 104
  MOs: 104
  Type: Restricted
  Has overlap: Yes
  Has basis set: Yes
  Valid: Yes
  Has active space: Yes
  Active Orbitals: α=6, β=6
  Has inactive space: Yes
  Inactive Orbitals: α=17, β=17
  Virtual Orbitals: α=81, β=81

[2026-04-02 17:15:35.349861] [info] [qdk:chemistry:algorithms:microsoft:active_space:valence_active_space] ValenceActiveSpaceSelector::Starting active space selection.
[2026-04-02 17:15:35.350027] [info] [qdk:chemistry:algorithms:microsoft:active_space:valence_active_space] ValenceActiveSpaceSelector::Selected active space of 6 orbitals: 17, 18, 19, 20, 21, 22


The next cell shows how to visualize the selected active orbitals.
The drop-down menu provides the ability to select different occupied and virtual orbitals in the active space to visualize their shapes, while the isovalue slider adjusts the surface representation of the orbitals for different electron density levels.


### Calculate the multi-configuration wavefunction for the active space

Once the active space has been selected, we are ready to solve the electronic structure problem (e.g., [Schrodinger's equation](https://en.wikipedia.org/wiki/Schr%C3%B6dinger_equation#Time-independent_equation)) more accurately than our initial SCF guess.
However, this requires two steps illustrated in this cell:

1. First, we need to construct the [Hamiltonian](https://en.wikipedia.org/wiki/Hamiltonian_(quantum_mechanics)), which provides the mathematical description of the energy and interactions of the electrons in the active space.

2. Second, we need to construct an improved estimate of the multi-configuration wavefunction and ground-state energy for this Hamiltonian.
The benzene diradical system in this demonstration is small enough that we can use a Complete Active Space [Configuration Interaction](https://en.wikipedia.org/wiki/Configuration_interaction) (CAS-CI) calculation to obtain the exact quantum mechanical energy and wavefunction for the active space.
However, for larger systems, the exact solution will not be feasible classically, and approximate methods such as [selected configuration interaction](https://arxiv.org/abs/2303.05688) or [density matrix renormalization group (DMRG)](https://en.wikipedia.org/wiki/Density_matrix_renormalization_group) are required.

Unlike the mean-field Hartree-Fock method, which approximates the wavefunction as a single [Slater determinant](https://en.wikipedia.org/wiki/Slater_determinant), these multi-configuration methods consider all possible electron configurations within the active space, capturing electron correlation effects.
By subtracting the mean-field Hartree-Fock energy from the correlated multi-configuration energy, we obtain the [correlation energy](https://en.wikipedia.org/wiki/Electronic_correlation) for this active space.

In [4]:
# Construct Hamiltonian in the active space and print its summary
hamiltonian_constructor = create("hamiltonian_constructor")
hamiltonian = hamiltonian_constructor.run(active_orbitals)
print("Active Space Hamiltonian:\n", hamiltonian.get_summary())

# Perform CASCI calculation to get the wavefunction and exact energy for the active space
mc = create("multi_configuration_calculator")
E_cas, wfn_cas = mc.run(
    hamiltonian, n_active_alpha_electrons=3, n_active_beta_electrons=3
)
print(f"CASCI energy is {E_cas:.3f} Hartree, and the electron correlation energy is {E_cas - E_hf:.3f} Hartree")

Active Space Hamiltonian:
 Hamiltonian Summary:
  Type: Hermitian
  Restrictedness: Restricted
  Active Orbitals: 6
  Total Orbitals: 104
  Core Energy: -223.094600
  Integral Statistics:
    One-body Integrals (alpha): 36 (larger than 0.000001: 6)
    Two-body Integrals (aaaa): 1296 (larger than 0.000001: 168)

[2026-04-02 17:15:37.073] [ci_solver] [info] [Selected CI Solver]:
[2026-04-02 17:15:37.073] [ci_solver] [info]   NDETS =    400, MATEL_TOL = 2.22045e-16, RES_TOL = 1.00000e-06, MAX_SUB =  200
[2026-04-02 17:15:37.084] [ci_solver] [info]   NNZ   =  22448, H_DUR     = 1.05001e+01 ms
[2026-04-02 17:15:37.084] [ci_solver] [info]   HMEM_LOC = 6.54e-04 GiB
[2026-04-02 17:15:37.084] [ci_solver] [info]   H_SPARSE = 1.40e+01%
[2026-04-02 17:15:37.084] [ci_solver] [info]   * Will generate identity guess
[2026-04-02 17:15:37.084] [davidson] [info] [Davidson Eigensolver]:
[2026-04-02 17:15:37.084] [davidson] [info]   N =    400, MAX_M =  200, RES_TOL = 1.00000e-06
[2026-04-02 17:15:37.084

## Loading the wavefunction onto a quantum computer

Now that we have calculated the multi-configuration wavefunction for the active space, we can generate a quantum circuit to prepare this state on a quantum computer.
However, not all parts of the multi-configuration wavefunction contribute equally to the overall state, creating an opportunity for optimization.

### Identifying the dominant configurations in the wavefunction

The first task is to understand the sparsity of the wavefunction:  how many configurations contribute significantly to the overall state?

This cell demonstrates how to analyze the wavefunction and identify the dominant configurations based on their amplitudes.

In [5]:
import numpy as np
from qdk.widgets import Histogram

# Plot top determinant weights from the CASCI wavefunction
NUM_DETERMINANTS = 10
print(f"Total determinants in the CASCI wavefunction:  {len(wfn_cas.get_active_determinants())}")
print(f"Plotting the top {NUM_DETERMINANTS} determinants by weight.")
top_configurations = wfn_cas.get_top_determinants(max_determinants=NUM_DETERMINANTS)
display(Histogram(bar_values={k.to_string(): np.abs(v)**2 for k, v in top_configurations.items()},))

Total determinants in the CASCI wavefunction:  400
Plotting the top 10 determinants by weight.


Reducing the wavefunction to these determinants allows us to optimize the computational requirements for loading the quantum computer with a state that has high overlap with the true wavefunction—an important metric for quantum algorithms like QPE.
However, this reduction of the wavefunction also changes our description of the quantum system, particularly its energy.
Therefore, for the purposes of benchmarking, we need to recalculate the energy of the truncated wavefunction classically to provide a reference for evaluating the accuracy of the quantum calculation.
This cell shows how to recalculate this energy.

In [6]:
# Get top 2 determinants from the CASCI wavefunction to form a sparse wavefunction
top_configurations = wfn_cas.get_top_determinants(max_determinants=8)

# Compute the reference energy of the sparse wavefunction
pmc_calculator = create("projected_multi_configuration_calculator")
E_sparse, wfn_sparse = pmc_calculator.run(hamiltonian, list(top_configurations.keys()))

print(f"Reference energy for top 2 determinants is {E_sparse:.6f} Hartree")

Reference energy for top 2 determinants is -229.397913 Hartree


In [7]:
for det, coeffs in zip(wfn_sparse.get_active_determinants(), wfn_sparse.get_coefficients()):
    alpha_str, beta_str = det.to_binary_strings(6)
    print(f"Determinant: {alpha_str[::-1]}{beta_str[::-1]}, Coefficient: {coeffs:.6f}")

Determinant: 000111000111, Coefficient: 0.787612
Determinant: 001011001011, Coefficient: -0.580092
Determinant: 010110010110, Coefficient: -0.099398
Determinant: 100101100101, Coefficient: -0.104064
Determinant: 101001101001, Coefficient: 0.075494
Determinant: 011010011010, Coefficient: 0.074410
Determinant: 100011001101, Coefficient: -0.074884
Determinant: 001101100011, Coefficient: -0.074884


In [ ]:
import pandas as pd
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state using the regular isometry method (Qiskit)
state_prep = create("state_prep", "qiskit_regular_isometry", transpile=False)
regular_isometry_circuit = state_prep.run(wfn_sparse)

# Visualize the regular isometry circuit
display(Circuit(regular_isometry_circuit.get_qsharp_circuit()))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(regular_isometry_circuit.estimate().logical_counts.items(), columns=['Logical Estimate', 'Counts'])
display(df)

### Loading the wavefunction using optimized state preparation methods

As the cell above illustrates, the general isometry method for state preparation can be very resource intensive—requiring thousands of fine rotations for this benzene diradical example.
However, we can optimize this process by taking advantage of the sparse multi-configuration wavefunction structure, generating much more efficient quantum circuits for state preparation.
The cell below demonstrates how the `qdk-chemistry` library can be used for optimized wavefunction loading, producing a circuit that is orders of magnitude more efficient than the general isometry method.

The underlying approach is based on a variation of the [sparse isometry method](https://quantum-journal.org/papers/q-2021-03-15-412/pdf/), with new updates specific to `qdk-chemistry` that avoid the use of multi-controlled gates (also challenging for near-term fault-tolerant quantum computers).

In [ ]:
import pandas as pd
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state via sparse isometry (GF2 + X)
state_prep = create("state_prep", "sparse_isometry_gf2x")
sparse_isometry_circuit = state_prep.run(wfn_sparse)

# Visualize the sparse isometry circuit
display(Circuit(sparse_isometry_circuit.get_qsharp_circuit(prune_classical_qubits=True)))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(
    sparse_isometry_circuit.estimate().logical_counts.items(), columns=['Logical Estimate', 'Counts'])
display(df)

[2026-04-02 17:15:40.744314] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Original matrix rank: 5
[2026-04-02 17:15:40.765453] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 6 identical to row 0, adding CX(0, 6)
[2026-04-02 17:15:40.774934] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 10 identical to row 4, adding CX(4, 10)
[2026-04-02 17:15:40.791613] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Eliminating 2 duplicate rows: [6, 10]
[2026-04-02 17:15:40.800593] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Final reduced matrix rank: 5


,Logical Estimate,Counts
0,numQubits,12
1,tCount,0
2,rotationCount,62
3,rotationDepth,62
4,cczCount,0
5,ccixCount,0
6,measurementCount,0


In [ ]:
from qdk_chemistry.data import (
    Ansatz,
    BasisSet,
    CanonicalFourCenterHamiltonianContainer,
    CasWavefunctionContainer,
    Configuration,
    Hamiltonian,
    Orbitals,
    OrbitalType,
    Shell,
    Structure,
    Wavefunction,
)

def create_test_basis_set(num_atomic_orbitals, name="test-basis", structure=None):
    """Create a test basis set with the specified number of atomic orbitals.

    Args:
        num_atomic_orbitals: Number of atomic orbitals to generate
        name: Name for the basis set
        structure: a structure to attach

    Returns:
        qdk_chemistry.data.BasisSet: A valid basis set for testing

    """
    shells = []
    atom_index = 0
    functions_created = 0

    # Create shells to reach the desired number of atomic orbitals
    while functions_created < num_atomic_orbitals:
        remaining = num_atomic_orbitals - functions_created

        if remaining >= 3:
            # Add a P shell (3 functions: Px, Py, Pz)
            exps = np.array([1.0, 0.5])
            coefs = np.array([0.6, 0.4])
            shell = Shell(atom_index, OrbitalType.P, exps, coefs)
            shells.append(shell)
            functions_created += 3
        elif remaining >= 1:
            # Add S shells for remaining functions (1 function each)
            for _ in range(remaining):
                exps = np.array([1.0])
                coefs = np.array([1.0])
                shell = Shell(atom_index, OrbitalType.S, exps, coefs)
                shells.append(shell)
                functions_created += 1
    if structure is not None:
        return BasisSet(name, shells, structure)
    return BasisSet(name, shells)


def create_test_orbitals(num_orbitals: int):
    """Helper: construct Orbitals immutably with identity coeffs and occupations.

    Occupations are set in restricted form as total occupancy per MO (0/1/2).
    """
    coeffs = np.eye(num_orbitals)
    basis_set = create_test_basis_set(num_orbitals)
    return Orbitals(coeffs, None, None, basis_set)


In [1]:
from random_wavefunction import generate_random_wavefunction

num_qubits = 26
seed = 42
wfn = generate_random_wavefunction(n_electrons=num_qubits//2, n_orbitals=num_qubits//2, n_dets=num_qubits, seed=seed)

TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. qdk_chemistry._core.data.Orbitals(arg0: qdk_chemistry._core.data.Orbitals)
    2. qdk_chemistry._core.data.Orbitals(coefficients: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, n]"], energies: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, 1]"] | None = None, ao_overlap: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, n]"] | None = None, basis_set: qdk_chemistry._core.data.BasisSet, indices: tuple[collections.abc.Sequence[typing.SupportsInt | typing.SupportsIndex], collections.abc.Sequence[typing.SupportsInt | typing.SupportsIndex]] | None = None)
    3. qdk_chemistry._core.data.Orbitals(coefficients_alpha: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, n]"], coefficients_beta: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, n]"], energies_alpha: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[m, 1]"]

In [ ]:
from sparse_isometry_binary_encoding import SparseIsometryBinaryEncodingStatePreparation

sp_select = SparseIsometryBinaryEncodingStatePreparation()
sp_select.settings().update("use_measurement_and", True)
select_circuit = sp_select.run(wfn_sparse)
display(Circuit(qsharp.circuit(select_circuit._qsharp_factory.program, *select_circuit._qsharp_factory.parameter.values(), generation_method=qsharp.CircuitGenerationMethod.Static)))

df = pd.DataFrame(
    qsharp.estimate(select_circuit._qsharp_factory.program, None, *select_circuit._qsharp_factory.parameter.values()).logical_counts.items(), columns=['Logical Estimate', 'Counts'])
display(df)

[2026-04-02 17:15:44.716994] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Original matrix rank: 5
[2026-04-02 17:15:44.727282] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 6 identical to row 0, adding CX(0, 6)
[2026-04-02 17:15:44.735311] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 10 identical to row 4, adding CX(4, 10)
[2026-04-02 17:15:44.744733] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Eliminating 2 duplicate rows: [6, 10]
[2026-04-02 17:15:44.751399] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Final reduced matrix rank: 5
[2026-04-02 17:15:44.784098] [info] [sparse_isometry_binary_encoding] Binary encoding produced 20 operations (20 encoded) using 0 ancillae for 12-qubit system with 8 determinants


,Logical Estimate,Counts
0,numQubits,14
1,tCount,7
2,rotationCount,7
3,rotationDepth,7
4,cczCount,7
5,ccixCount,0
6,measurementCount,6


### Estimating the energy on a quantum computer (simulator)

Finally, we need to generate the measurement circuits required to estimate the energy of the prepared multi-configuration wavefunction on a quantum computer.
The cell below demonstrates how to generate these measurement circuits using the `qdk-chemistry` library and how to use the QDK simulator to execute them.

This cell provides a set budget of measurements ("shots") to be evenly divided between the measurement circuits.
The measurement process is probabilistic, so we obtain a distribution of results from each circuit.
These distributions are then combined to produce a final energy estimate, along with an uncertainty (variance) as reported below.
The uncertainty is directly related to the number of shots used in the measurement process:  more shots lead to lower uncertainty.

In [ ]:
# Estimate energy using the optimized circuit and the qubit Hamiltonian
estimator = create("energy_estimator", algorithm_name="qdk")
circuit_executor = create("circuit_executor", algorithm_name="qdk_full_state_simulator")
energy_results, simulation_data = estimator.run(
    circuit=sparse_isometry_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    total_shots=1500000,
)

# Print statistic for measured energy
energy_mean = energy_results.energy_expectation_value + hamiltonian.get_core_energy()
energy_stddev = np.sqrt(energy_results.energy_variance)
print(
    f"Estimated energy from quantum circuit: {energy_mean:.3f} ± {energy_stddev:.3f} Hartree"
)

# Print comparison with reference energy
print(f"Difference from reference energy: {energy_mean - E_sparse} Hartree")

In [ ]:
# Diagnostic: inspect what the circuit parameters contain
params = circuit._qsharp_factory.parameter
print(f"numQubits: {params['numQubits']}")
print(f"numAncilla: {params['numAncilla']}")
print(f"rowMap: {params['rowMap']}")
print(f"stateVector length: {len(params['stateVector'])}")
print(f"expansionOps count: {len(params['expansionOps'])}")
print(f"binaryEncodingOps count: {len(params['binaryEncodingOps'])}")

# Show expansion ops
print("\n--- Expansion Ops ---")
for i, op in enumerate(params['expansionOps'][:10]):
    print(f"  [{i}] {op['name']} qubits={op['qubits']}")
if len(params['expansionOps']) > 10:
    print(f"  ... ({len(params['expansionOps'])} total)")

# Show binary encoding ops
print("\n--- Binary Encoding Ops ---")
for i, op in enumerate(params['binaryEncodingOps'][:10]):
    print(f"  [{i}] {op['name']} qubits={op['qubits']} lookupData={len(op.get('lookupData', [])) if op.get('lookupData') else 0} rows")
if len(params['binaryEncodingOps']) > 10:
    print(f"  ... ({len(params['binaryEncodingOps'])} total)")

# Check GF2+X operations directly
print("\n--- GF2+X result operations (from gf2x_result) ---")
gf2x_result, _ = sp._perform_gf2x(
    [beta_str[::-1] + alpha_str[::-1]
     for det in wfn_sparse.get_active_determinants()
     for alpha_str, beta_str in [det.to_binary_strings(num_orbitals)]],
    wfn_sparse.get_coefficients()
)
print(f"Total GF2+X operations: {len(gf2x_result.operations)}")
op_types = {}
for op in gf2x_result.operations:
    op_types[op[0]] = op_types.get(op[0], 0) + 1
print(f"Operation types: {op_types}")
print(f"row_map: {gf2x_result.row_map}")

numQubits: 12
numAncilla: 0
rowMap: [0, 2, 1]
stateVector length: 8
expansionOps count: 28
binaryEncodingOps count: 20

--- Expansion Ops ---
  [0] CX qubits=[4, 11]
  [1] CX qubits=[4, 9]
  [2] CX qubits=[4, 8]
  [3] CX qubits=[4, 7]
  [4] CX qubits=[4, 3]
  [5] CX qubits=[4, 1]
  [6] CX qubits=[3, 11]
  [7] CX qubits=[3, 9]
  [8] CX qubits=[3, 5]
  [9] CX qubits=[3, 4]
  ... (28 total)

--- Binary Encoding Ops ---
  [0] SELECT qubits=[0, 2, 1, 3] lookupData=8 rows
  [1] CX qubits=[3, 1] lookupData=0 rows
  [2] SELECT qubits=[0, 2, 1, 3] lookupData=8 rows
  [3] CX qubits=[3, 2] lookupData=0 rows
  [4] SELECT qubits=[0, 2, 1, 3] lookupData=8 rows
  [5] CX qubits=[3, 1] lookupData=0 rows
  [6] CX qubits=[3, 2] lookupData=0 rows
  [7] CX qubits=[3, 0] lookupData=0 rows
  [8] SWAP qubits=[2, 1] lookupData=0 rows
  [9] SWAP qubits=[0, 4] lookupData=0 rows
  ... (20 total)

--- GF2+X result operations (from gf2x_result) ---


[2026-04-02 15:47:10.039960] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Original matrix rank: 5
[2026-04-02 15:47:10.062632] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 6 identical to row 0, adding CX(0, 6)
[2026-04-02 15:47:10.074259] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Found duplicate row 10 identical to row 4, adding CX(4, 10)
[2026-04-02 15:47:10.083060] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Eliminating 2 duplicate rows: [6, 10]
[2026-04-02 15:47:10.099927] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] Final reduced matrix rank: 5
Total GF2+X operations: 28
Operation types: {'cx': 28}
row_map: [0, 2, 1, 3, 4]
